In [2]:
import plotly.express as px
import pandas as pd
import numpy as np
import pickle
import os

In [3]:
with open('dict_sofipos_sector.pkl', 'rb') as archivo:
    dict_sofipos = pickle.load(archivo)

In [4]:
dict_sofipos = dict_sofipos['fintech']
dict_sofipos.keys()

dict_keys(['Inf_cartera', 'Inf_cartera_E1E2', 'Inf_cartera_E3', 'EPRC', 'Capt_trad', 'Prest_bank', 'CC', 'CG', 'Res_acum', 'Res_ejer', 'Ing_int', 'Ing_int_E1E2', 'Ing_int_E3', 'Ind_financieros', 'Castigos'])

In [22]:
df_cartera = dict_sofipos['Inf_cartera_E3']

In [6]:
df_cartera['periodo'] = pd.to_datetime(df_cartera['periodo'], format= '%Y-%m')

In [7]:
def plot_ln(df : pd.DataFrame, 
            serie : str, 
            titulo: str, 
            categorias : str,
            sofipos : list,
            log : bool,
            doble_eje : bool = True,
            eje : str | None = None,
            inicio : str | None = None,
            mostrar : bool = True):

        import pandas as pd
        import numpy as np
        import plotly.express as px
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots

        # Define el inicio de las series
        if pd.isna(inicio) == False:

            df = df[df['periodo'] >= inicio] 

        # Asegura orden cronológico para que el "último valor" sea correcto
        df = df.sort_values('periodo')

        if log == True:
            # Convierte a logaritmo e la serie
            df[serie] = df[serie].astype(float)
            df[f'{serie}_ln'] = np.log(df[serie])
        
            # Selecciona las sofipos 
            df = df[df['sofipo'].isin(sofipos)]

            y_col = f'{serie}_ln'

        else:
            # Selecciona las sofipos 
            df = df[df['sofipo'].isin(sofipos)]
            y_col = serie
        
        fig_px = px.line(
            df, 
            x="periodo", 
            y=y_col, 
            title= titulo,
            color=categorias)

        if doble_eje:
            # Construye la figura con doble eje
            fig = make_subplots(specs=[[{"secondary_y": True}]])

            for trace in fig_px.data:
                es_total = trace.name in ['Total SOFIPOS', 'Fintech']
                fig.add_trace(trace, secondary_y=not es_total)

            fig.update_layout(
                title=titulo,
                xaxis=dict(
                    title=None,
                    tickformat="%Y",
                    dtick="M12",
                    hoverformat="%Y-%m")
                )
            
            fig.update_yaxes(title_text="Sector SOFIPOS", secondary_y=False)
            fig.update_yaxes(title_text="Otras SOFIPOS", secondary_y=True)

            # Líneas de referencia en y = 100 para cada eje
            fig.add_hline(y=100, line_dash="dash", line_color="black", secondary_y=False)
            fig.add_hline(y=100, line_dash="dash", line_color="darkblue", secondary_y=True)

        else:
            # Figura de un solo eje
            fig = fig_px

            fig.update_layout(
                title=titulo,
                xaxis=dict(
                    title=None,
                    tickformat="%Y",
                    dtick="M12",
                    hoverformat="%Y-%m")
                )

            fig.update_yaxes(
                side="right",
                showticklabels=True,
                title_text= eje
            )

            # Línea de referencia en y = 100
            fig.add_hline(y=100, line_dash="dash", line_color="black")

        # Leyenda dentro del gráfico, esquina superior izquierda
        fig.update_layout(
            legend=dict(
                title=dict(text="SOFIPO"),
                x=0.01,
                y=0.99,
                xanchor="left",
                yanchor="top",
                bgcolor="rgba(255,255,255,0.6)"
            )
        )

        # Agrega el último valor y fecha a la leyenda de cada serie
        def actualizar_leyenda(trace):
            x_vals = trace.x
            y_vals = trace.y

            if len(x_vals) == 0:
                return

            ultimo_x = pd.to_datetime(x_vals[-1]).strftime('%Y-%m')
            ultimo_y = y_vals[-1]

            trace.update(name=f"{trace.name} | {ultimo_y:.2f} ({ultimo_x})")

        fig.for_each_trace(actualizar_leyenda)

        if mostrar:
            fig.show()

        return fig

In [ ]:
df_cartera['sofipo'].unique()

In [8]:
plt_sector = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Total SOFIPOS', 'Fintech'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01')

In [9]:
plt_sofipo = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Fincomún', 'Tamazula', 'Libertad', 'Crediclub'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01')

In [21]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

panel = make_subplots(rows=2, cols=1, 
                      subplot_titles=("SECTOR", "SOFIPO"),
                      vertical_spacing=0.08)

plt_sector = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Total SOFIPOS', 'Fintech'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01',
                mostrar=False)

plt_sofipo = plot_ln(df = df_cartera,
                serie= 'Cartera de crédito_i_b100',
                titulo= '',
                categorias= 'sofipo',
                sofipos= ['Fincomún', 'Tamazula', 'Libertad', 'Crediclub'],
                doble_eje = False,
                log=False,
                eje = 'Cartera Total de Crédito (2023-01 = 100)',
                inicio= '2022-01',
                mostrar=False)

# Traces del primer gráfico → leyenda 1
for trace in plt_sector.data:
    trace.legend = "legend"
    panel.add_trace(trace, row=1, col=1)

# Traces del segundo gráfico → leyenda 2
for trace in plt_sofipo.data:
    trace.legend = "legend2"
    panel.add_trace(trace, row=2, col=1)

# Replica el formato del eje X en ambas filas
panel.update_xaxes(title=None, tickformat="%Y", dtick="M12", hoverformat="%Y-%m",
                    row=1, col=1)
panel.update_xaxes(title=None, tickformat="%Y", dtick="M12", hoverformat="%Y-%m",
                    row=2, col=1)

# Replica el título y posición del eje Y en ambas filas
panel.update_yaxes(side="right", showticklabels=True,
                    title_text='Cartera Total de Crédito (2023-01 = 100)',
                    row=1, col=1)
panel.update_yaxes(side="right", showticklabels=True,
                    title_text='Cartera Total de Crédito (2023-01 = 100)',
                    row=2, col=1)

# Replica las líneas de referencia en y=100 en ambas filas
panel.add_hline(y=100, line_dash="dash", line_color="black", row=1, col=1)
panel.add_hline(y=100, line_dash="dash", line_color="black", row=2, col=1)

# Layout general del panel
panel.update_layout(
    title="Cartera Total de Crédito",
    width=1200,
    height=900,
    legend=dict(
        title=dict(text="SOFIPO"),
        x=0.01, y=0.99,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        tracegroupgap=1 
    ),
    legend2=dict(
        title=dict(text="SOFIPO"),
        x=0.01, y=0.45,   # ajusta 'y' para que caiga junto al segundo subplot
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        tracegroupgap=1 
    )
)
panel.show()

In [ ]:
df_cartera['periodo'] = pd.to_datetime(df_cartera['periodo'].astype(str), format='%Y-%m-%d').dt.to_period('M')

In [ ]:
df_cartera[df_cartera['periodo'] >= '2022-02']